# Entrainement YOLOX - comptage de vis (Devoq AI)

Notebook **cle en main** pour Google Colab (GPU gratuit).

Etapes : installer YOLOX -> uploader le dataset -> convertir YOLO->COCO -> fine-tuner YOLOX-S -> exporter en ONNX -> telecharger `yolox.onnx`.

**Avant de commencer** : Menu `Execution > Modifier le type d'execution > GPU` (T4).

Le `.onnx` produit est directement compatible avec le microservice IA (`AI_DETECTOR_MODE=yolox_onnx`).

## 1. Verifier le GPU

In [ ]:
!nvidia-smi

## 2. Installer YOLOX et ses dependances

In [ ]:
%cd /content
!git clone https://github.com/Megvii-BaseDetection/YOLOX.git
%cd /content/YOLOX
!pip install -q -r requirements.txt
!pip install -q -v -e .
!pip install -q onnx onnxruntime onnx-simplifier loguru

## 3. Uploader le dataset

Sur ton PC, compresse le dossier `datasets/processed` en un zip nomme **`processed.zip`** (clic droit > Envoyer vers > Dossier compresse). Il doit contenir `images/train`, `images/val`, `labels/train`, `labels/val`.

Lance la cellule puis selectionne `processed.zip`.

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # selectionne processed.zip
zip_name = next(iter(uploaded))
os.makedirs('/content/processed', exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content/processed')

# Localise le dossier qui contient images/ et labels/ (le zip peut avoir un niveau processed/ en plus)
import glob
root = '/content/processed'
if not os.path.isdir(os.path.join(root, 'images')):
    cand = glob.glob('/content/processed/**/images', recursive=True)
    assert cand, 'Dossier images/ introuvable dans le zip'
    root = os.path.dirname(cand[0])
print('Dataset racine :', root)
print('train images :', len(glob.glob(os.path.join(root, 'images/train/*'))))
print('val images   :', len(glob.glob(os.path.join(root, 'images/val/*'))))

## 4. Convertir YOLO -> COCO (format attendu par YOLOX)

In [ ]:
import json, os, glob
from PIL import Image

IMG_EXT = ('.jpg', '.jpeg', '.png', '.webp')
DATA = '/content/datasets/coco'

def convert(split_in, split_out):
    img_dir = os.path.join(root, 'images', split_in)
    lbl_dir = os.path.join(root, 'labels', split_in)
    out_img = os.path.join(DATA, split_out)
    os.makedirs(out_img, exist_ok=True)
    os.makedirs(os.path.join(DATA, 'annotations'), exist_ok=True)
    images, annotations = [], []
    categories = [{'id': 1, 'name': 'screw', 'supercategory': 'screw'}]
    ann_id, img_id = 1, 1
    import shutil
    for p in sorted(glob.glob(os.path.join(img_dir, '*'))):
        if not p.lower().endswith(IMG_EXT):
            continue
        w, h = Image.open(p).size
        fname = os.path.basename(p)
        shutil.copy2(p, os.path.join(out_img, fname))
        images.append({'id': img_id, 'file_name': fname, 'width': w, 'height': h})
        lbl = os.path.join(lbl_dir, os.path.splitext(fname)[0] + '.txt')
        if os.path.exists(lbl):
            for line in open(lbl):
                line = line.strip()
                if not line:
                    continue
                _, cx, cy, bw, bh = map(float, line.split())
                x, y, bw_px, bh_px = (cx - bw / 2) * w, (cy - bh / 2) * h, bw * w, bh * h
                annotations.append({'id': ann_id, 'image_id': img_id, 'category_id': 1,
                                    'bbox': [x, y, bw_px, bh_px], 'area': bw_px * bh_px, 'iscrowd': 0})
                ann_id += 1
        img_id += 1
    out_json = os.path.join(DATA, 'annotations', f'instances_{split_out}.json')
    json.dump({'images': images, 'annotations': annotations, 'categories': categories}, open(out_json, 'w'))
    print(f'{split_out}: {len(images)} images, {len(annotations)} boites -> {out_json}')

convert('train', 'train2017')
convert('val', 'val2017')
# Si le split val est vide, on reutilise train pour permettre l'evaluation
if len(glob.glob(os.path.join(DATA, 'val2017', '*'))) == 0:
    print('[!] val vide -> copie de train comme val (a eviter sur un vrai projet)')
    convert('train', 'val2017')

## 5. Ecrire la config d'experience YOLOX (1 classe : screw)

In [ ]:
%%writefile /content/yolox_screw_exp.py
import os
from yolox.exp import Exp as MyBaseExp

class Exp(MyBaseExp):
    def __init__(self):
        super().__init__()
        self.depth = 0.33
        self.width = 0.50
        self.num_classes = 1
        self.data_dir = os.environ.get('YOLOX_DATA_DIR', '/content/datasets/coco')
        self.train_ann = 'instances_train2017.json'
        self.val_ann = 'instances_val2017.json'
        self.train_name = 'train2017'
        self.val_name = 'val2017'
        self.input_size = (640, 640)
        self.test_size = (640, 640)
        self.max_epoch = int(os.environ.get('YOLOX_MAX_EPOCH', 100))
        self.data_num_workers = 2
        self.eval_interval = 5
        self.warmup_epochs = 5
        self.no_aug_epochs = 15
        self.print_interval = 10
        self.mosaic_prob = 1.0
        self.mixup_prob = 1.0
        self.hsv_prob = 1.0
        self.flip_prob = 0.5
        self.exp_name = 'yolox_screw'

## 6. Telecharger le checkpoint pre-entraine YOLOX-S (point de depart du fine-tuning)

In [ ]:
%cd /content/YOLOX
!wget -q https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_s.pth -O yolox_s.pth
!ls -lh yolox_s.pth

## 7. Entrainer

`-b` = taille de batch (8 convient a un T4 16 Go ; baisse a 4 si memoire insuffisante). Regle `YOLOX_MAX_EPOCH` selon la taille du dataset.

In [ ]:
import os
os.environ['YOLOX_DATA_DIR'] = '/content/datasets/coco'
os.environ['YOLOX_MAX_EPOCH'] = '100'
!python tools/train.py -f /content/yolox_screw_exp.py -d 1 -b 8 --fp16 -o -c yolox_s.pth

## 8. Exporter en ONNX

`--decode_in_inference` est **indispensable** : il integre le decodage dans le modele, de sorte que la sortie soit au format `[1, N, 5+classes]` (cx, cy, w, h, objectness, classes) attendu par le detecteur `yolox_onnx` du projet.

In [ ]:
!python tools/export_onnx.py \
  -f /content/yolox_screw_exp.py \
  -c YOLOX_outputs/yolox_screw/best_ckpt.pth \
  --output-name yolox.onnx \
  --decode_in_inference

# Verifie la forme de sortie : on attend [1, N, 6] (4 box + 1 obj + 1 classe)
import onnxruntime as ort
s = ort.InferenceSession('yolox.onnx', providers=['CPUExecutionProvider'])
print('input :', s.get_inputs()[0].name, s.get_inputs()[0].shape)
print('output:', s.get_outputs()[0].name, s.get_outputs()[0].shape)

## 9. Telecharger le modele

In [ ]:
from google.colab import files
files.download('/content/YOLOX/yolox.onnx')

## 10. Deployer

1. Remplace `models/yolox.onnx` dans le projet par le fichier telecharge.
2. Redemarre l'ai-service en mode YOLOX (voir `training/README.md`).
3. Teste le comptage dans l'app.